# Лабораторная D5: формула, сетка и предельная ценность

90 минут внутри блока D. Сначала выполни письменный вывод D5. Выбери ядро Masters2027, затем Run All. До заполнения функции статус будет `PENDING`, это не ошибка установки и не зачёт.

Здесь проверяется модель `u=8x+6y-(x²+y²)/2`, `x,y≥0`, `2x+y≤m`. Поле `shadow` означает производную функции ценности, при нуле — правую. Не вводи произвольный множитель из множества KKT.

Шаги: 25 минут — реализовать формулу; 20 — сравнить с сеткой; 20 — графики и производная; 15 — интерпретация; 10 — чистый запуск и сохранение. Код использует только локальные расчёты и не скачивает данные.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv
from pathlib import Path

def utility(x, y):
    return 8*x + 6*y - (x*x + y*y)/2

In [ ]:
def consumer(m):
    """Конечное m>=0 -> dict(x=..., y=..., shadow=...).
    При отрицательном или неконечном m: ValueError.
    shadow при m=0 — правая производная функции ценности.
    """
    raise NotImplementedError("Сначала выведи режимы D5, затем заполни функцию")

## Независимая сетка

Перебираются не только точки бюджетной прямой: после насыщения оптимум лежит внутри бюджета. При каждом x перебирается весь допустимый диапазон y. Сетка приближённая; увеличение числа узлов уменьшает ошибку, но не заменяет аналитическое доказательство.

In [ ]:
def grid_best(m, n=201):
    if not np.isfinite(m) or m < 0:
        raise ValueError("m must be finite and nonnegative")
    xs = np.linspace(0, m/2, n)
    fractions = np.linspace(0, 1, n)
    X = np.broadcast_to(xs[:, None], (n, n))
    Y = (m - 2*xs[:, None]) * fractions[None, :]
    scores = utility(X, Y)
    i, j = np.unravel_index(np.argmax(scores), scores.shape)
    return float(X[i,j]), float(Y[i,j]), float(scores[i,j])

try:
    consumer(0)
    READY = True
except NotImplementedError:
    READY = False
    print("PENDING: функция ещё не заполнена. Это не успешная проверка.")

In [ ]:
rows = []
if READY:
    for m in [0., 0.5, 1.999, 2., 2.001, 5., 12., 21.999, 22., 22.001, 26.]:
        ans = consumer(m)
        x, y, shadow = [float(ans[k]) for k in ("x", "y", "shadow")]
        assert np.isfinite([x,y,shadow]).all(), "Неконечный результат"
        assert x >= -1e-9 and y >= -1e-9 and 2*x+y <= m+1e-9, "Недопустимая корзина"
        assert shadow >= -1e-9, "Ценность расширения бюджета неотрицательна"
        gx, gy, gv = grid_best(m)
        val = utility(x,y)
        assert val >= gv-1e-8, "Сетка нашла лучшую допустимую корзину"
        rows.append([m,x,y,shadow,val,gx,gy,gv,val-gv])
    for row in rows:
        print("m=%7.3f x=%8.4f y=%8.4f shadow=%7.4f value-grid=%9.6f" %
              (row[0],row[1],row[2],row[3],row[8]))
    print("Числовая сверка завершена. Письменные доказательства оцениваются отдельно.")

## Производная и рисунки

До запуска предскажи границы режимов и вид трёх графиков. В нуле используй правую разность. В точках смены режима сравни односторонние производные: гладкость ценности не означает гладкость спроса.

Проверь значения `grid_best(12,n)` для n=51,101,201,401. Запиши, как меняется разрыв ценности и почему найденная сеточная корзина может не совпадать с точной.

In [ ]:
if READY:
    ms = np.linspace(0, 28, 281)
    results = [consumer(float(m)) for m in ms]
    xs = np.array([a['x'] for a in results])
    ys = np.array([a['y'] for a in results])
    shadows = np.array([a['shadow'] for a in results])
    vals = utility(xs,ys)
    fig, axes = plt.subplots(1,3,figsize=(13,3.5))
    axes[0].plot(ms,xs,label='x*'); axes[0].plot(ms,ys,label='y*'); axes[0].legend()
    axes[1].plot(ms,vals); axes[1].set_title('Value function')
    axes[2].plot(ms,shadows); axes[2].set_title('Marginal value')
    for ax in axes:
        ax.set_xlabel('income m'); ax.grid(alpha=.2)
    plt.tight_layout(); plt.show()
    def value(m):
        a=consumer(m)
        return utility(a['x'],a['y'])
    for m in [0., 1., 2., 12., 22., 26.]:
        h=1e-5
        right=(value(m+h)-value(m))/h
        left=None if m==0 else (value(m)-value(m-h))/h
        print('m',m,'left',left,'right',right,'reported',consumer(m)['shadow'])
    for n in [51,101,201,401]:
        print('grid size',n, 'best',grid_best(12.,n))

## Сохранение и письменный вывод

Запусти следующую ячейку, затем приложи CSV и сохранённый notebook. В комментарии ответь: почему поиск только по бюджетной прямой неверен после насыщения; почему сетка не доказывает глобальность; почему `shadow(0)` — не любой допустимый множитель; что изменилось при удвоении плотности сетки.

После сдачи попытки можно открыть `lab_reference.py`. Это учебный ключ, не код, который ноутбук незаметно вызывает вместо твоего решения.

In [ ]:
if READY:
    candidates=[Path.cwd(), *Path.cwd().parents]
    root=next((p for p in candidates if (p/'course.json').exists()), None)
    if root is None:
        raise RuntimeError("Открой notebook внутри скачанного репозитория")
    destination=root/'work'/'first_week'/'lab_results.csv'
    destination.parent.mkdir(parents=True,exist_ok=True)
    with destination.open('w',newline='',encoding='utf-8') as f:
        writer=csv.writer(f)
        writer.writerow(['m','x','y','shadow','value','grid_x','grid_y','grid_value','value_gap'])
        writer.writerows(rows)
    print(destination)
else:
    print("CSV не создан: функция ещё не реализована.")